In [ ]:
import warnings
warnings.filterwarnings("ignore")
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_wine, load_digits
from sklearn.model_selection import train_test_split, GridSearchCV, learning_curve
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, ConfusionMatrixDisplay, roc_curve, auc

In [ ]:
models = {
    "SVM linear": SVC(kernel="linear"),
    "SVM poly": SVC(kernel="poly"),
    "SVM rbf": SVC(kernel="rbf"),
    "SVM sigmoid": SVC(kernel="sigmoid"),
    "MLP": MLPClassifier(max_iter=500, random_state=42),
    "RF": RandomForestClassifier(random_state=42),
}

svm_grid = {"C": [0.1, 1, 10]}
grids = {
    "SVM linear": svm_grid,
    "SVM poly": svm_grid,
    "SVM rbf": svm_grid,
    "SVM sigmoid": svm_grid,
    "MLP": {"solver": ["sgd"], "momentum": [0.5, 0.9], "max_iter": [300, 500], "learning_rate_init": [0.01, 0.1]},
    "RF": {"n_estimators": [50, 100, 200]},
}

In [ ]:
def evaluate(name, model, experiment):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    print(name, experiment, "Accuracy:", accuracy)
    print(classification_report(y_test, y_pred))
    ConfusionMatrixDisplay.from_predictions(y_test, y_pred, cmap="Blues")
    plt.title(f"{name} {experiment}")
    plt.show()
    results.append([name, experiment, accuracy])

In [ ]:
def run_all(experiment):
    for name in models:
        evaluate(name, models[name], experiment + " default")
        grid = GridSearchCV(models[name], grids[name], cv=5)
        evaluate(name, grid, experiment + " tuned")
        print(grid.best_params_)

In [ ]:
dataset = load_wine()
X, y = dataset.data, dataset.target
results = []

for test_size in [0.5, 0.4, 0.3, 0.2]:
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=42, stratify=y)
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)
    run_all(f"test {test_size}")

In [ ]:
y_true = pd.get_dummies(y_test).values.ravel()
for name in models:
    model = models[name]
    model.fit(X_train, y_train)
    if "SVM" in name:
        scores = model.decision_function(X_test)
    else:
        scores = model.predict_proba(X_test)
    fpr, tpr, thresholds = roc_curve(y_true, scores.ravel())
    plt.plot(fpr, tpr, label=f"{name} AUC = {auc(fpr, tpr):.3f}")
plt.xlabel("False positive rate")
plt.ylabel("True positive rate")
plt.title("ROC curve")
plt.legend()
plt.show()

In [ ]:
for name in models:
    sizes, train_scores, val_scores = learning_curve(models[name], X_train, y_train, cv=5)
    plt.plot(sizes, train_scores.mean(axis=1), label="training accuracy")
    plt.plot(sizes, val_scores.mean(axis=1), label="validation accuracy")
    plt.plot(sizes, 1 - val_scores.mean(axis=1), label="validation loss")
    plt.title(f"{name} learning curve")
    plt.xlabel("Training rows")
    plt.legend()
    plt.show()

plt.plot(models["MLP"].loss_curve_)
plt.title("MLP loss curve")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.show()

In [ ]:
pca = PCA(n_components=0.95)
X_train = pca.fit_transform(X_train)
X_test = pca.transform(X_test)
print("Features after PCA:", X_train.shape[1])
run_all("PCA")

table = pd.DataFrame(results, columns=["classifier", "experiment", "accuracy"])
table.pivot_table(index="experiment", columns="classifier", values="accuracy").round(3)

In [ ]:
dataset = load_digits()
X, y = dataset.data, dataset.target
results = []

for test_size in [0.5, 0.4, 0.3, 0.2]:
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=test_size, random_state=42, stratify=y)
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)
    run_all(f"test {test_size}")

In [ ]:
y_true = pd.get_dummies(y_test).values.ravel()
for name in models:
    model = models[name]
    model.fit(X_train, y_train)
    if "SVM" in name:
        scores = model.decision_function(X_test)
    else:
        scores = model.predict_proba(X_test)
    fpr, tpr, thresholds = roc_curve(y_true, scores.ravel())
    plt.plot(fpr, tpr, label=f"{name} AUC = {auc(fpr, tpr):.3f}")
plt.xlabel("False positive rate")
plt.ylabel("True positive rate")
plt.title("ROC curve")
plt.legend()
plt.show()

In [ ]:
for name in models:
    sizes, train_scores, val_scores = learning_curve(models[name], X_train, y_train, cv=5)
    plt.plot(sizes, train_scores.mean(axis=1), label="training accuracy")
    plt.plot(sizes, val_scores.mean(axis=1), label="validation accuracy")
    plt.plot(sizes, 1 - val_scores.mean(axis=1), label="validation loss")
    plt.title(f"{name} learning curve")
    plt.xlabel("Training rows")
    plt.legend()
    plt.show()

plt.plot(models["MLP"].loss_curve_)
plt.title("MLP loss curve")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.show()

In [ ]:
pca = PCA(n_components=0.95)
X_train = pca.fit_transform(X_train)
X_test = pca.transform(X_test)
print("Features after PCA:", X_train.shape[1])
run_all("PCA")

table = pd.DataFrame(results, columns=["classifier", "experiment", "accuracy"])
table.pivot_table(index="experiment", columns="classifier", values="accuracy").round(3)